# ShopPulse: E-Commerce Sales Analytics Platform
## Notebook 02: Production Data Cleaning & Validation Pipeline

### Objectives:
1. Ingest raw data extract (`data/raw/raw_ecommerce_data.csv`)
2. Remove duplicate orders and exact duplicate records
3. Strip whitespace discrepancies in categorical attributes
4. Impute missing customer names and payment methods using relational heuristics
5. Convert date strings to standardized datetime formats and extract analytical temporal features
6. Enforce financial integrity constraints (`sales`, `cost`, `profit`, `profit_margin_pct`)
7. Export the production-grade dataset to `data/processed/cleaned_ecommerce_data.csv`


In [ ]:
import pandas as pd
import numpy as np
import os
import sys

# Ensure root directory is on python path
sys.path.append('..')
from src.data_cleaning import DataCleaner

print("DataCleaner imported successfully.")


### 1. Initialize and Run Cleaning Pipeline


In [ ]:
cleaner = DataCleaner(
    raw_filepath='../data/raw/raw_ecommerce_data.csv',
    output_filepath='../data/processed/cleaned_ecommerce_data.csv'
)

df_clean = cleaner.run_pipeline()
print(f"Processed dataset ready: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} columns.")


### 2. Audit Trail & Transformation Metrics


In [ ]:
print("=== Data Cleaning Audit Log ===")
for metric, val in cleaner.audit_log.items():
    print(f"{metric}: {val}")


### 3. Post-Cleaning Validation Checks


In [ ]:
# Validation 1: Zero Missing Values
remaining_nulls = df_clean.isnull().sum().sum()
assert remaining_nulls == 0, f"Error: Found {remaining_nulls} null values!"
print(" Validation Check 1 Passed: 0 Missing values across all columns.")

# Validation 2: Zero Duplicate Order IDs
duplicate_orders = df_clean.duplicated(subset=['order_id']).sum()
assert duplicate_orders == 0, f"Error: Found {duplicate_orders} duplicate order IDs!"
print(" Validation Check 2 Passed: 0 Duplicate order IDs.")

# Validation 3: Financial Mathematical Integrity
# Check Sales = Quantity * Unit_Price * (1 - Discount)
computed_sales = (df_clean['quantity'] * df_clean['unit_price'] * (1.0 - df_clean['discount'])).round(2)
sales_diff = (df_clean['sales'] - computed_sales).abs().max()
assert sales_diff <= 0.05, f"Error: Sales calculation discrepancy of {sales_diff}"
print(f" Validation Check 3 Passed: Financial Sales formula verified (Max delta = {sales_diff}).")

# Validation 4: Profit = Sales - Cost
computed_profit = (df_clean['sales'] - df_clean['cost']).round(2)
profit_diff = (df_clean['profit'] - computed_profit).abs().max()
assert profit_diff <= 0.05, f"Error: Profit calculation discrepancy of {profit_diff}"
print(f" Validation Check 4 Passed: Profit formula verified (Max delta = {profit_diff}).")


### 4. Cleaned Dataset Sample & Schema Inspection


In [ ]:
df_clean.head(5)


### 5. Summary Statistics of Cleaned Data


In [ ]:
summary_table = pd.DataFrame({
    'Metric': ['Total Transactions', 'Total Revenue ($)', 'Total Gross Profit ($)', 'Overall Margin (%)',
               'Unique Customers', 'Unique Products', 'Average Order Value ($)', 'Average Discount (%)'],
    'Value': [
        f"{len(df_clean):,}",
        f"${df_clean['sales'].sum():,.2f}",
        f"${df_clean['profit'].sum():,.2f}",
        f"{(df_clean['profit'].sum() / df_clean['sales'].sum() * 100):.2f}%",
        f"{df_clean['customer_id'].nunique():,}",
        f"{df_clean['product_id'].nunique():,}",
        f"${df_clean['sales'].mean():,.2f}",
        f"{(df_clean['discount'].mean() * 100):.2f}%"
    ]
})
summary_table
